In [5]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from google import genai
from google.genai import types
client = genai.Client()

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

Q1. How many lesson pages are in the dataset?

In [7]:
len(documents)

72

___

Q2. Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

How does the agentic loop keep calling the model until it stops?

What's the filename of the first result?

In [8]:
from minsearch import Index


index = Index(
    text_fields = ['content'],
    keyword_fields = ['filename']
)

index.fit(documents)

In [35]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(question, num_results=5)

A. 01-agentic-rag/lessons/14-agentic-loop.md

___

Q3.

Build a RAG over the index from Q2 and answer the query:

"How does the agentic loop keep calling the model until it stops?"

How many input (prompt) tokens did we send to the model for this request?


Most LLM APIs report token usage on the response object (e.g. response.usage.input_tokens / prompt_tokens). We'll read the input tokens from there.



In the RAG Helper class, llm returns only the text. Modify it to return the whole response, and change rag to return both the answer and usage (as a tuple or create a small dataclass for that).

In [11]:
import importlib.util

spec = importlib.util.spec_from_file_location('rag_helper', 'rag_helper.py')
rag_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rag_module)

rager = rag_module.RAGBase(index, client)
question = "How does the agentic loop keep calling the model until it stops?"

response = rager.llm(rager.build_prompt(question, rager.search(question, num_results=5)))

print("Answer:", response.text)
print("Input tokens:", response.usage_metadata.prompt_token_count)

Answer: The agentic loop keeps calling the model until it stops by wrapping the process of sending messages, running tools, and appending results in a `while` loop.

Here's how it works:

1.  **Initial Call**: The loop starts by making an API call to the model, providing the `instructions` (developer prompt), `user` question, and available `tools`.
2.  **Process Model Response**: The model's response is appended to the `messages` history.
3.  **Check for Function Calls**: The code iterates through the items in the model's `response.output`.
    *   If an item is a `function_call` (e.g., `search`), the `make_call` helper function executes the specified tool with the arguments provided by the model. The output of the tool call is then appended back to the `messages` history. A flag `has_function_calls` is set to `True`.
    *   If an item is a `message`, it means the model has generated a text response, which is printed.
4.  **Loop Continuation/Termination**:
    *   The `has_function_ca

___

Q4. Chunking

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [14]:
len(chunks)

295

In [15]:
chunk_index = Index(text_fields=['content'], keyword_fields=['filename'])
chunk_index.fit(chunks)

rager_chunks = rag_module.RAGBase(chunk_index, client)

answer, usage = rager_chunks.rag("How does the agentic loop keep calling the model until it stops?")

q3_tokens = 7948
q4_tokens = usage.prompt_token_count
difference = q3_tokens - q4_tokens

print(f"Q3 tokens: {q3_tokens}")
print(f"Q4 tokens: {q4_tokens}")
print(f"Fewer tokens: {difference}")

Q3 tokens: 7948
Q4 tokens: 2600
Fewer tokens: 5348


___

In [ ]:
from toyaikit.tools import Tools

search_count = 0

def search(query: str) -> str:
    """Search the course chunks for relevant information."""
    global search_count
    search_count += 1
    results = chunk_index.search(query, num_results=5)
    return '\n'.join([f"{r['filename']}: {r['content'][:200]}" for r in results])

# Create toyaikit Tools instance and add our search function
tools_kit = Tools()
tools_kit.add_tool(search)

# Get the tools definition from toyaikit and clean up for genai
tools_list = tools_kit.get_tools()

# Clean schema to remove toyaikit-specific fields
def clean_schema(schema):
    """Remove additionalProperties and other fields genai doesn't support"""
    clean = {}
    if "type" in schema:
        clean["type"] = schema["type"]
    if "properties" in schema:
        clean["properties"] = schema["properties"]
    if "required" in schema:
        clean["required"] = schema["required"]
    return clean

# Create tool definition for genai from toyaikit
tool_defs = [
    types.Tool(function_declarations=[
        types.FunctionDeclaration(
            name=tool["name"],
            description=tool["description"],
            parameters=clean_schema(tool["parameters"])
        )
    ])
    for tool in tools_list
]

instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

messages = [
    types.Content(role="user", parts=[types.Part(text="How does the agentic loop work, and how is it different from plain RAG?")])
]

config = types.GenerateContentConfig(
    system_instruction=instructions,
    tools=tool_defs
)

max_iterations = 5
for i in range(max_iterations):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=messages,
        config=config
    )
    
    # Check if model wants to use tools
    if response.candidates[0].content.parts[0].function_call:
        func_call = response.candidates[0].content.parts[0].function_call
        tool_name = func_call.name
        args = dict(func_call.args)
        
        # Execute the tool directly (we know it's search)
        if tool_name == "search":
            result = search(args["query"])
        else:
            result = "Unknown tool"
        
        # Add model response and tool result to messages
        messages.append(response.candidates[0].content)
        messages.append(
            types.Content(role="user", parts=[
                types.Part(text=f"Tool result: {result}")
            ])
        )
    else:
        # Model generated final answer
        break

print(response.text)
print(f"\nSearch calls: {search_count}")

The agentic loop is a core concept in AI agents, representing a `while` loop that iteratively calls the model. You build this loop by hand, and it allows for repeated interactions and refinement. The `LoopResult` of an agentic loop contains all messages from the conversation, token counts, and the computed cost based on token usage.

In contrast, a plain RAG (Retrieval Augmented Generation) pipeline typically involves three steps:
1. Retrieval of relevant information.
2. Augmentation of the prompt with the retrieved information.
3. Generation of a response by the Language Model.

While both RAG and the agentic loop involve an LLM, the agentic loop introduces a dynamic, iterative process where the agent can make multiple calls to the model, possibly incorporating function calls and refining its understanding or actions over several turns. This is different from the generally more linear, single-pass nature of a plain RAG pipeline.

Search calls: 3
